In [33]:
# =============================================================================
# DKTC 전처리 파이프라인 (최종 정리본)
#
# 흐름:
#   1. DKTC 원본 로드 (raw)
#   2. 020 일반대화 로드 (raw) + 자연 필터 + 플래그 마킹
#   3. 합치기 (둘 다 raw 상태)
#   4. 전체에 동일한 clean_text + normalize 적용
#   5. 이상치/저품질 필터링
#   6. train/val 분할 (증강 전)
#   7. 증강 (학습셋만)
#   8. 저장
#
# 출력: data/train_processed.csv, data/val_processed.csv
# =============================================================================
 
import pandas as pd
import numpy as np
import re
import random
from pathlib import Path
from sklearn.model_selection import train_test_split
 
random.seed(42)
np.random.seed(42)

In [34]:
# =============================================================================
# 함수 정의
# =============================================================================
 
def clean_text(text: str) -> str:
    """한국어 대화체 텍스트 정제"""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)                      # 반복문자 축소
    text = re.sub(r'([ㄱ-ㅎㅏ-ㅣ])\1{2,}', r'\1\1', text)             # 자모 반복 축소
    text = re.sub(r'[^\w\s가-힣a-zA-Z0-9.,!?~\n]', ' ', text)        # 특수문자 제거
    text = re.sub(r'[ \t]+', ' ', text)                               # 연속 공백
    text = re.sub(r'\n+', '\n', text)                                 # 연속 줄바꿈
    return text.strip()
 
 
def normalize_conversation(text: str) -> str:
    """대화 턴 구분을 [턴] 토큰으로 변환"""
    if not isinstance(text, str):
        return ""
    turns = [t.strip() for t in text.split('\n') if t.strip()]
    return " [턴] ".join(turns)
 
 
def extract_meta_features(text: str) -> dict:
    """대화 구조적 특성 추출"""
    turns = [t.strip() for t in text.split('\n') if t.strip()]
    lengths = [len(t) for t in turns]
    return {
        "n_turns": len(turns),
        "total_chars": sum(lengths),
        "avg_turn_len": np.mean(lengths) if lengths else 0,
        "max_turn_len": max(lengths) if lengths else 0,
    }
 
 
def augment_turn_shuffle(text: str) -> str:
    """인접 턴 1쌍 swap"""
    turns = [t.strip() for t in text.split('\n') if t.strip()]
    if len(turns) < 3:
        return text
    idx = random.randint(0, len(turns) - 2)
    turns[idx], turns[idx + 1] = turns[idx + 1], turns[idx]
    return '\n'.join(turns)
 
 
def augment_random_deletion(text: str, p: float = 0.1) -> str:
    """각 턴에서 단어를 확률적으로 삭제"""
    turns = [t.strip() for t in text.split('\n') if t.strip()]
    new_turns = []
    for turn in turns:
        words = turn.split()
        if len(words) <= 2:
            new_turns.append(turn)
            continue
        new_words = [w for w in words if random.random() > p]
        if not new_words:
            new_words = [random.choice(words)]
        new_turns.append(' '.join(new_words))
    return '\n'.join(new_turns)
 
 
def augment_turn_drop(text: str) -> str:
    """랜덤 턴 하나 삭제"""
    turns = [t.strip() for t in text.split('\n') if t.strip()]
    if len(turns) <= 2:
        return text
    drop_idx = random.randint(0, len(turns) - 1)
    turns.pop(drop_idx)
    return '\n'.join(turns)
 
 
def augment_data(df: pd.DataFrame, target_per_class: int = None) -> pd.DataFrame:
    """클래스별 증강하여 균형 맞춤"""
    augmenters = [augment_turn_shuffle, augment_random_deletion, augment_turn_drop]
    class_counts = df["label"].value_counts()
    if target_per_class is None:
        target_per_class = class_counts.max()
 
    augmented_rows = []
    for label in df["label"].unique():
        label_df = df[df["label"] == label]
        needed = target_per_class - len(label_df)
        if needed <= 0:
            continue
        for _ in range(needed):
            row = label_df.sample(1).iloc[0].copy()
            aug_fn = random.choice(augmenters)
            row["conversation"] = aug_fn(row["conversation"])
            row["conversation_clean"] = clean_text(row["conversation"])
            row["conversation_norm"] = normalize_conversation(row["conversation_clean"])
            row["id"] = -1
            augmented_rows.append(row)
 
    if augmented_rows:
        return pd.concat([df, pd.DataFrame(augmented_rows)], ignore_index=True)
    return df.copy()

In [35]:
# =============================================================================
# 1. DKTC 원본 로드
# =============================================================================
print("=" * 60)
print("1. DKTC 로드")
print("=" * 60)
 
dktc_df = pd.read_csv("data/train.csv")  # 로컬 저장본 사용
# GitHub에서 직접 받을 경우:
# url = "https://raw.githubusercontent.com/tunib-ai/DKTC/main/data/train.csv"
# dktc_df = pd.read_csv(url, header=None, names=["id", "label_name", "conversation"])
# dktc_df = dktc_df[dktc_df["id"] != "idx"].reset_index(drop=True)
 
# 라벨 확인
label_map = {"협박 대화": 0, "갈취 대화": 1, "직장 내 괴롭힘 대화": 2, "기타 괴롭힘 대화": 3}
if "label" not in dktc_df.columns:
    dktc_df["label"] = dktc_df["label_name"].map(label_map).astype(int)
 
dktc_df["source"] = "dktc"
print(f"  DKTC: {len(dktc_df)}개")
print(dktc_df["label_name"].value_counts().to_string())

1. DKTC 로드
  DKTC: 3950개
label_name
기타 괴롭힘 대화      1094
갈취 대화           981
직장 내 괴롭힘 대화     979
협박 대화           896


In [36]:
# =============================================================================
# 2. 020 일반대화 로드 + 자연 필터 + 플래그 마킹
# =============================================================================
print(f"\n{'=' * 60}")
print("2. 020 일반대화 로드 + 필터링")
print("=" * 60)
 
normal_full = pd.read_csv("data/normal_020_full.csv")
normal_full["conversation"] = normal_full["conversation"].str.replace(" [SEP] ", "\n")
normal_full["source"] = "020"
 
print(f"  전체: {len(normal_full)}개")
 
# 자연 필터링 (DKTC와 분포 맞춤)
normal_filtered = normal_full[
    (normal_full["n_turns"] <= 14) & (normal_full["total_chars"] <= 450)
].copy()
print(f"  자연 필터 후: {len(normal_filtered)}개")
 
# ┌─────────────────────────────────────────────────────────┐
# │  여기서 원하는 만큼 잘라 쓰기                           │
# │  N_NORMAL 값만 바꾸면 됨                                │
# └─────────────────────────────────────────────────────────┘
N_NORMAL = 1000  # ← 실험 A=500, B=1000, C=2000
 
if N_NORMAL > len(normal_filtered):
    print(f"  ⚠️ 요청({N_NORMAL})이 필터 결과({len(normal_filtered)})보다 큼 → 전체 사용")
    normal_sampled = normal_filtered.copy()
else:
    normal_sampled = normal_filtered.sample(n=N_NORMAL, random_state=42)
 
print(f"  샘플링: {len(normal_sampled)}개")


2. 020 일반대화 로드 + 필터링
  전체: 87331개
  자연 필터 후: 39802개
  샘플링: 1000개


In [37]:
# =============================================================================
# 3. 합치기 (둘 다 raw 상태)
# =============================================================================
print(f"\n{'=' * 60}")
print("3. 데이터 합치기")
print("=" * 60)
 
raw_df = pd.concat([
    dktc_df[["id", "label_name", "conversation", "label", "source"]],
    normal_sampled[["id", "label_name", "conversation", "label", "source"]]
], ignore_index=True)
 
print(f"  합산: {len(raw_df)}개")
print(f"  출처별: dktc={len(raw_df[raw_df['source']=='dktc'])}, 020={len(raw_df[raw_df['source']=='020'])}")
print(raw_df["label_name"].value_counts().to_string())


3. 데이터 합치기
  합산: 4950개
  출처별: dktc=3950, 020=1000
label_name
기타 괴롭힘 대화      1094
일반 대화          1000
갈취 대화           981
직장 내 괴롭힘 대화     979
협박 대화           896


In [38]:
# =============================================================================
# 4. 전체에 동일한 정제 + 정규화 + 피처 추출
# =============================================================================
print(f"\n{'=' * 60}")
print("4. 텍스트 정제 + 피처 추출")
print("=" * 60)
 
raw_df["conversation_clean"] = raw_df["conversation"].apply(clean_text)
raw_df["conversation_norm"] = raw_df["conversation_clean"].apply(normalize_conversation)
 
meta = raw_df["conversation"].apply(extract_meta_features).apply(pd.Series)
raw_df = pd.concat([raw_df, meta], axis=1)
 
print(f"\n  정제 후 클래스별 통계:")
print(raw_df.groupby("label_name")[["n_turns", "total_chars", "avg_turn_len"]].mean().round(1).to_string())


4. 텍스트 정제 + 피처 추출

  정제 후 클래스별 통계:
             n_turns  total_chars  avg_turn_len
label_name                                     
갈취 대화           10.5        204.7          19.4
기타 괴롭힘 대화       10.2        199.0          19.4
일반 대화           11.6        228.5          19.6
직장 내 괴롭힘 대화     10.4        226.2          21.7
협박 대화           10.3        234.6          22.6


In [39]:
# =============================================================================
# 5. 이상치 / 저품질 데이터 필터링
# =============================================================================
print(f"\n{'=' * 60}")
print("5. 이상치 필터링")
print("=" * 60)
 
before = len(raw_df)
raw_df = raw_df[raw_df["conversation_clean"].str.len() > 0]
raw_df = raw_df[~((raw_df["n_turns"] <= 1) & (raw_df["total_chars"] < 10))]
raw_df = raw_df.drop_duplicates(subset=["conversation_clean"], keep="first")
raw_df = raw_df.drop_duplicates(subset=["conversation_norm"], keep="first")
 
print(f"  {before} → {len(raw_df)} ({before - len(raw_df)}개 제거)")
print(raw_df["label_name"].value_counts().to_string())


5. 이상치 필터링
  4950 → 4845 (105개 제거)
label_name
기타 괴롭힘 대화      1010
일반 대화          1000
갈취 대화           973
직장 내 괴롭힘 대화     970
협박 대화           892


In [40]:
# =============================================================================
# 6. train/val 분할 (증강 전 — 데이터 누출 방지)
# =============================================================================
print(f"\n{'=' * 60}")
print("6. train/val 분할")
print("=" * 60)
 
train_df, val_df = train_test_split(
    raw_df,
    test_size=0.2,
    stratify=raw_df["label"],
    random_state=42
)
 
print(f"  학습: {len(train_df)}")
print(f"  검증: {len(val_df)}")


6. train/val 분할
  학습: 3876
  검증: 969


In [41]:
# =============================================================================
# 7. 클래스 균형 조정 + 증강 (학습셋만)
# =============================================================================
print(f"\n{'=' * 60}")
print("7. 클래스 균형 + 증강")
print("=" * 60)
 
# 일반대화가 과다하면 다운샘플링
threat_max = train_df[train_df["label"] != 4]["label"].value_counts().max()
normal_count = len(train_df[train_df["label"] == 4])
 
if normal_count > threat_max * 2:
    normal_subset = train_df[train_df["label"] == 4].sample(n=threat_max, random_state=42)
    train_df = pd.concat([train_df[train_df["label"] != 4], normal_subset], ignore_index=True)
    target = threat_max
    print(f"  일반대화 다운샘플링: {normal_count} → {threat_max}")
else:
    target = int(train_df["label"].value_counts().median())
 
print(f"  증강 목표: 클래스당 {target}개")
 
train_aug = augment_data(train_df, target_per_class=target)
 
print(f"\n  증강 후:")
print(train_aug["label_name"].value_counts().to_string())


7. 클래스 균형 + 증강
  증강 목표: 클래스당 778개

  증강 후:
label_name
기타 괴롭힘 대화      808
일반 대화          800
협박 대화          778
갈취 대화          778
직장 내 괴롭힘 대화    778


In [42]:
# =============================================================================
# 8. 최종 저장
# =============================================================================
print(f"\n{'=' * 60}")
print("8. 저장")
print("=" * 60)
 
output_cols = ["conversation_norm", "label", "label_name", "n_turns", "total_chars"]
 
train_final = train_aug[output_cols].rename(columns={"conversation_norm": "text"})
val_final = val_df[output_cols].rename(columns={"conversation_norm": "text"})
 
train_final = train_final[train_final["text"].str.len() > 0].reset_index(drop=True)
val_final = val_final[val_final["text"].str.len() > 0].reset_index(drop=True)
 
Path("data").mkdir(exist_ok=True)
train_final.to_csv("data/train_processed_260317_n_1000.csv", index=False)
val_final.to_csv("data/val_processed_260317_n_1000.csv", index=False)
 
print(f"  학습: {len(train_final)}개")
print(f"  검증: {len(val_final)}개")
print(f"\n  학습 클래스 분포:")
print(train_final["label_name"].value_counts().to_string())
print(f"\n  검증 클래스 분포:")
print(val_final["label_name"].value_counts().to_string())


8. 저장
  학습: 3942개
  검증: 969개

  학습 클래스 분포:
label_name
기타 괴롭힘 대화      808
일반 대화          800
협박 대화          778
갈취 대화          778
직장 내 괴롭힘 대화    778

  검증 클래스 분포:
label_name
기타 괴롭힘 대화      202
일반 대화          200
갈취 대화          195
직장 내 괴롭힘 대화    194
협박 대화          178


In [43]:
# =============================================================================
# 9. 품질 리포트
# =============================================================================
print(f"\n{'=' * 60}")
print("9. 품질 리포트")
print("=" * 60)
 
print(f"\n  텍스트 길이 (글자 수):")
for label_name in sorted(train_final["label_name"].unique()):
    lengths = train_final[train_final["label_name"] == label_name]["text"].str.len()
    print(f"    {label_name}: mean={lengths.mean():.0f}, median={lengths.median():.0f}, "
          f"min={lengths.min()}, max={lengths.max()}")
 
print(f"\n  max_length 설정 가이드:")
print(f"    텍스트 최대 길이: {train_final['text'].str.len().max()} 글자")
print(f"    95퍼센타일: {train_final['text'].str.len().quantile(0.95):.0f} 글자")
 
for ml in [128, 256, 512]:
    cov = (train_final["text"].str.len() <= ml).mean() * 100
    print(f"    max_length={ml} → 커버리지 {cov:.1f}%")
 
print(f"\n  저장 경로: data/train_processed.csv, data/val_processed.csv")
print(f"  텍스트 컬럼: 'text'  |  라벨 컬럼: 'label' (0~4)")
print(f"{'=' * 60}")


9. 품질 리포트

  텍스트 길이 (글자 수):
    갈취 대화: mean=249, median=222, min=81, max=715
    기타 괴롭힘 대화: mean=248, median=222, min=77, max=918
    일반 대화: mean=282, median=270, min=128, max=499
    직장 내 괴롭힘 대화: mean=271, median=248, min=77, max=821
    협박 대화: mean=282, median=255, min=93, max=886

  max_length 설정 가이드:
    텍스트 최대 길이: 918 글자
    95퍼센타일: 451 글자
    max_length=128 → 커버리지 1.8%
    max_length=256 → 커버리지 55.0%
    max_length=512 → 커버리지 97.1%

  저장 경로: data/train_processed.csv, data/val_processed.csv
  텍스트 컬럼: 'text'  |  라벨 컬럼: 'label' (0~4)


- 클래스당 778~808으로 거의 균등
- 글자 수 분포도 일반대화(mean=282)와 위협 대화(mean=248~282)가 비슷
- max_length 결정 필요 
    - 256 커버리지 55.0% 속도빠름 비고절반 잘림,위험
    - 512 커버리지 97.1% 속도 중간 